# 02a Byte BPE Generation

This notebook trains the byte-level BPE tokenizer for the rebuilt workflow.

The Byte-BPE tokenizer is the simplest tokenizer in the comparison set. It does not encode glycan-specific biological structure directly, so it serves as a useful baseline for the more structured tokenizers trained later.

**Main outputs**
- tokenizer files saved under `tokenizers/byte_bpe/<setting_label>/`
- `vocab.json`
- `merges.txt`
- `tokenizer_config_summary.json`
- optionally `inspection_preview.csv`


## Setup and user settings

The first code cell performs the small amount of bootstrap work that must happen before shared project helpers can be imported. The second code cell contains the values you may need to edit before running the notebook, especially `PROJECT_ROOT`, the notebook-specific parameters, and the overwrite setting.


In [ ]:
# Standard library imports used for the initial Colab bootstrap.
import subprocess
import sys
from pathlib import Path

from google.colab import drive

# Mount Google Drive so the notebook can read project data and save outputs.
drive.mount('/content/drive')

# These public GitHub settings identify the repository that stores the notebook
# helpers. They normally do not need to change unless the project is moved to a
# different repository.
GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
GITHUB_REF = 'main'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = Path('/content') / REPO_NAME

# The repository must exist locally before the notebook can import shared
# helper modules from the src package.
if not REPO_DIR.exists():
    print(f'Cloning repository from {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'Repository already exists at {REPO_DIR}.')

# Add the repository root to the Python import path so the shared setup helper
# can be imported in the next cell.
repo_dir_str = str(REPO_DIR)
if repo_dir_str not in sys.path:
    sys.path.insert(0, repo_dir_str)

from src.notebook_setup import bootstrap_notebook_from_settings

print(f'Repository bootstrap directory: {REPO_DIR}')


In [ ]:
from pathlib import Path

from src.tokenizer_notebook_utils import build_tokenizer_output_paths, build_tokenizer_paths

# Update PROJECT_ROOT if your Drive project folder has a different name or
# location. This is the main path value that should be checked before running.
PROJECT_ROOT = Path('/content/drive/MyDrive/ProjectRoot')

# This notebook trains on the training split created earlier in the workflow.
TRAIN_SPLIT_FILENAME = 'train.txt'

# If True, the notebook may replace previously saved tokenizer artifacts in the
# target output folder. If False, the notebook will stop before overwriting files.
OVERWRITE_EXISTING_OUTPUTS = False

# These settings control the Byte-BPE vocabulary size and the minimum token
# frequency required for merge learning.
VOCAB_SIZE = 300
MIN_FREQUENCY = 2

# Save a lightweight CSV preview of the tokenizer sanity check when True.
SAVE_INSPECTION_PREVIEW = True

SETTING_LABEL = f'v{VOCAB_SIZE}_m{MIN_FREQUENCY}'
TOKENIZER_FAMILY = 'byte_bpe'

# Use the shared setup helper to update the local repository copy and validate
# the project root while keeping the editable settings visible in this notebook.
ctx = bootstrap_notebook_from_settings(
    notebook_name='02a_byte_bpe_gen',
    project_root=PROJECT_ROOT,
    github_owner=GITHUB_OWNER,
    repo_name=REPO_NAME,
    github_ref=GITHUB_REF,
    repo_dir=REPO_DIR,
    require_drive=False,
    require_repo_sync=True,
)
project_root = ctx.project_root

# Build the standard paths used by this tokenizer notebook.
paths = build_tokenizer_paths(
    project_root=project_root,
    tokenizer_family=TOKENIZER_FAMILY,
    setting_label=SETTING_LABEL,
    train_split_filename=TRAIN_SPLIT_FILENAME,
)
output_paths = build_tokenizer_output_paths(
    tokenizer_output_dir=paths['tokenizer_output_dir'],
    include_inspection_preview=SAVE_INSPECTION_PREVIEW,
    include_merges_file=True,
)

train_data_path = paths['train_data_path']
tokenizer_output_dir = paths['tokenizer_output_dir']
config_summary_path = paths['config_summary_path']
inspection_preview_path = paths['inspection_preview_path']

tokenizer_output_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root: {project_root}')
print(f'Training data path: {train_data_path}')
print(f'Tokenizer output directory: {tokenizer_output_dir}')
print(f'Setting label: {SETTING_LABEL}')


## Train, save, and inspect the tokenizer

The next code cell validates the input/output paths, trains the Byte-BPE tokenizer, saves the tokenizer artifacts, and records a short configuration summary.

The final code cell reloads the saved tokenizer, shows a small inspection table, and optionally saves that preview as a CSV.


In [ ]:
import json

from tokenizers import ByteLevelBPETokenizer
from transformers import PreTrainedTokenizerFast

from src.notebook_utils import require_existing_path, validate_output_paths
from src.tokenizer_notebook_utils import save_tokenizer_summary

# Verify that the notebook can find the training split before continuing.
require_existing_path(train_data_path, 'Tokenizer training split')

# Enforce the shared overwrite policy before any tokenizer artifacts are saved.
validate_output_paths(
    output_paths=output_paths,
    overwrite_existing_outputs=OVERWRITE_EXISTING_OUTPUTS,
)

print(f'Training byte-level BPE tokenizer (vocab={VOCAB_SIZE}, min_frequency={MIN_FREQUENCY})...')

# Train a Byte-BPE tokenizer directly on the glycan training strings.
tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=[str(train_data_path)],
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=['<s>', '<pad>', '</s>', '<unk>', '<mask>'],
)

# Save the raw tokenizer model files such as vocab.json and merges.txt.
tokenizer.save_model(str(tokenizer_output_dir))

# Wrap the trained backend tokenizer in a Hugging Face fast tokenizer so later
# notebooks can load it through the standard Transformers interface.
hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer._tokenizer,
    bos_token='<s>',
    eos_token='</s>',
    unk_token='<unk>',
    pad_token='<pad>',
    mask_token='<mask>',
)

hf_tokenizer.save_pretrained(str(tokenizer_output_dir))

saved_files = sorted(path.name for path in tokenizer_output_dir.iterdir())
tokenizer_summary = {
    'tokenizer_family': TOKENIZER_FAMILY,
    'setting_label': SETTING_LABEL,
    'vocab_size': VOCAB_SIZE,
    'min_frequency': MIN_FREQUENCY,
    'train_data_path': str(train_data_path),
    'tokenizer_output_dir': str(tokenizer_output_dir),
    'saved_files': saved_files,
}

save_tokenizer_summary(tokenizer_summary, config_summary_path)

print('Tokenizer training complete.')
print(f'Tokenizer saved to: {tokenizer_output_dir}')


In [ ]:
from IPython.display import display
from transformers import PreTrainedTokenizerFast

from src.tokenizer_notebook_utils import (
    build_tokenizer_inspection_preview,
    load_training_sequences_for_tokenizer,
    save_inspection_preview,
)

# Reload the saved tokenizer from disk so the sanity check uses the same files
# that later notebooks will load.
loaded_tokenizer = PreTrainedTokenizerFast.from_pretrained(str(tokenizer_output_dir))
train_sequences = load_training_sequences_for_tokenizer(train_data_path)
inspection_df = build_tokenizer_inspection_preview(
    tokenizer=loaded_tokenizer,
    sequences=train_sequences,
    num_samples=3,
    max_display_tokens=30,
)

display(inspection_df)

print(f'Loaded vocabulary size: {len(loaded_tokenizer)}')
print(f'Mask token: {loaded_tokenizer.mask_token}')
print(f'Pad token: {loaded_tokenizer.pad_token}')

if SAVE_INSPECTION_PREVIEW:
    saved_inspection_path = save_inspection_preview(
        inspection_df=inspection_df,
        output_path=inspection_preview_path,
    )
    print(f'Inspection preview saved to: {saved_inspection_path}')
else:
    print('Inspection preview saving is disabled for this run.')
